In [ ]:
import os
import warnings

warnings.filterwarnings('ignore')
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # Suppress INFO, WARNING, and ERROR logs from TF/absl

import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, GlobalAveragePooling2D, RandomBrightness, RandomContrast, RandomZoom, RandomRotation, RandomFlip # type:ignore
from tensorflow.keras.applications.convnext import ConvNeXtXLarge, preprocess_input
from tensorflow.keras import Sequential # type:ignore
from sklearn.metrics import accuracy_score
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.optimizers import Adam # type:ignore
from tensorflow.keras.regularizers import l2 # type: ignore
from tensorflow.keras.preprocessing import image 
from tensorflow.keras.callbacks import EarlyStopping # type: ignore
from tensorflow.keras import Input # type:ignore

In [2]:
Convolution_layer = ConvNeXtXLarge(
    weights = "imagenet",
    include_top = False,
    input_shape = (180,180,3)
)

Convolution_layer.trainable = False

In [3]:
train = "C:\\Users\\sulem\\OneDrive\\Desktop\\Codes\\ML\\Datasets\\Birds Species\\train"
test = "C:\\Users\\sulem\\OneDrive\\Desktop\\Codes\\ML\\Datasets\\Birds Species\\valid"

def ImageTransformation(file):

    generator = tf.keras.utils.image_dataset_from_directory(
        directory = file,
        labels = "inferred",
        label_mode = "int",
        batch_size = 33,
        image_size = (180,180)
    )

    class_names = generator.class_names   

    def Transformation(img, label):
        img = preprocess_input(img)
        return img, label

    pixels = generator.map(Transformation)

    return pixels, class_names

train_pixels, train_classes = ImageTransformation(train)
test_pixels, test_classes = ImageTransformation(test)

print(train_classes == test_classes)   # should print True
print(len(train_classes))
print(len(test_classes))

def dataset_to_numpy(dataset):

    images = []
    labels = []

    for img_batch, label_batch in dataset:
        images.append(img_batch.numpy())
        labels.append(label_batch.numpy())

    X = np.concatenate(images, axis=0)
    y = np.concatenate(labels, axis=0)

    return X, y

trainx, trainy = dataset_to_numpy(train_pixels)
testx, testy = dataset_to_numpy(test_pixels)

print(trainx.shape, trainy.shape)
print(testx.shape, testy.shape)

Found 3189 files belonging to 20 classes.
Found 100 files belonging to 20 classes.
True
20
20
(3189, 180, 180, 3) (3189,)
(100, 180, 180, 3) (100,)


In [4]:
def build_model(meta):
    n_classes = meta["n_classes_"]  # pulled automatically from your y data - no hardcoding

    m = Sequential()
    m.add(Input(shape=(180, 180, 3))) 

    m.add(RandomFlip("horizontal"))
    m.add(RandomRotation(0.04))
    m.add(RandomZoom(0.12))
    m.add(RandomContrast(0.15))
    m.add(RandomBrightness(0.15))

    m.add(Convolution_layer)
    m.add(GlobalAveragePooling2D()) # GlobalAveragePooling is ConvNeXt's native pooling strategy.
    m.add(Dense(128, activation="relu", kernel_regularizer=l2(0.01)))
    m.add(BatchNormalization())
    m.add(Dropout(0.4))
    m.add(Dense(n_classes, activation="softmax"))  

    m.compile(loss="sparse_categorical_crossentropy", optimizer=Adam(learning_rate=0.001), metrics=["accuracy"])
    return m

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,                  # give it a few epochs of no improvement before stopping
    restore_best_weights=True    # roll back to the best epoch, not the last one
)

m = KerasClassifier(model=build_model, epochs=8, batch_size=33, verbose=1, validation_split=0.3, callbacks=[early_stop])

history = m.fit(trainx,trainy);

Epoch 1/8
68/68 ━━━━━━━━━━━━━━━━━━━━ 2331s 34s/step - accuracy: 0.9117 - loss: 2.4011 - val_accuracy: 0.9875 - val_loss: 1.6012
Epoch 2/8
68/68 ━━━━━━━━━━━━━━━━━━━━ 2322s 34s/step - accuracy: 0.9821 - loss: 1.2062 - val_accuracy: 0.9927 - val_loss: 0.8519
Epoch 3/8
68/68 ━━━━━━━━━━━━━━━━━━━━ 2291s 34s/step - accuracy: 0.9888 - loss: 0.6637 - val_accuracy: 0.9885 - val_loss: 0.5020
Epoch 4/8
68/68 ━━━━━━━━━━━━━━━━━━━━ 2584s 38s/step - accuracy: 0.9888 - loss: 0.3975 - val_accuracy: 0.9781 - val_loss: 0.3599
Epoch 5/8
68/68 ━━━━━━━━━━━━━━━━━━━━ 2324s 34s/step - accuracy: 0.9789 - loss: 0.3158 - val_accuracy: 0.9822 - val_loss: 0.2810
Epoch 6/8
68/68 ━━━━━━━━━━━━━━━━━━━━ 2638s 39s/step - accuracy: 0.9875 - loss: 0.2365 - val_accuracy: 0.9854 - val_loss: 0.2099
Epoch 7/8
68/68 ━━━━━━━━━━━━━━━━━━━━ 2725s 40s/step - accuracy: 0.9857 - loss: 0.2004 - val_accuracy: 0.9916 - val_loss: 0.1781
Epoch 8/8
68/68 ━━━━━━━━━━━━━━━━━━━━ 2338s 34s/step - accuracy: 0.9892 - loss: 0.1734 - val_accuracy: 0.

In [5]:
predy = m.predict(testx) 
print("Accuarcy Score:", accuracy_score(testy, predy))

4/4 ━━━━━━━━━━━━━━━━━━━━ 79s 17s/step
Accuarcy Score: 1.0


In [6]:
def predict_image(img_path, model, class_names, image_size=(180, 180)):

    img = image.load_img(img_path, target_size=image_size)
    img_array = image.img_to_array(img)

    img_array = tf.reshape(img_array, (1, 180, 180, 3))  # Add batch dimension
    img_array = preprocess_input(img_array)               # Same normalization as Transformation function

    prediction = model.predict(img_array)

    predicted_index = prediction[0]
    label = class_names[predicted_index]

    print("Prediction:", label)
    return label

In [7]:
final_test = "C:\\Users\\sulem\\OneDrive\\Desktop\\Codes\\ML\\Datasets\\Birds Species\\Final_test"

for filename in os.listdir(final_test):
    img_path = os.path.join(final_test, filename)
    predict_image(img_path, m, train_classes)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 707ms/step
Prediction: ABBOTTS BABBLER
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 719ms/step
Prediction: ABBOTTS BOOBY
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 785ms/step
Prediction: ABYSSINIAN GROUND HORNBILL
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 771ms/step
Prediction: AFRICAN CROWNED CRANE
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 709ms/step
Prediction: AFRICAN EMERALD CUCKOO
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 717ms/step
Prediction: AFRICAN FIREFINCH
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 791ms/step
Prediction: AFRICAN OYSTER CATCHER
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 792ms/step
Prediction: AFRICAN PIED HORNBILL
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 863ms/step
Prediction: AFRICAN PYGMY GOOSE
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 721ms/step
Prediction: ALBATROSS
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 722ms/step
Prediction: ALBERTS TOWHEE
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 716ms/step
Prediction: ALEXANDRINE PARAKEET
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 732ms/step
Prediction: ALPINE CHOUGH
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 795ms/step
Prediction: ALTAMIRA YELLOWTHROAT
1/1 ━━━━━━━━━━━━━━━

In [9]:
last_test = "C:\\Users\\sulem\\OneDrive\\Desktop\\Codes\\ML\\Datasets\\Birds Species\\Last_test\\kestrel_americano.jpg"

predict_image(last_test, m, train_classes)

1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step
Prediction: AMERICAN KESTREL


'AMERICAN KESTREL'